# PushT imitation learning: MLP, LSTM, and Transformer
**Google Colab starter notebook · verified 4 November 2026**

Based on Maryam Sharifzadeh and Navraj Kambo's UBC EECE 568 proposal, *Comparing MLP, LSTM, and Transformer Policies for Robot Imitation Learning on PushT*, found in the local course repository (`doc/EECE568_Proposal.pdf`).

This notebook loads Hugging Face parquet with pandas, inspects and validates episodes, creates reproducible train/validation/test splits, and trains the three proposed behavior-cloning models. Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. CPU also works. Download the generated `pusht_run` folder before the Colab session expires.

**Schema correction:** the verified `lerobot/pusht` v3 snapshot contains 206 episodes / 25,650 frames at 10 Hz. `observation.state` is only the 2D pusher position; `action` is a 2D target position. The proposal's expected 16 block/keypoint values are **absent**. Images live in separate MP4s, not parquet. These are therefore partially observed **pusher-only starter policies**, not the proposal's full pusher-plus-block experiment. No block state is fabricated, and actions, rewards, success flags, or identifiers are never input features.

The dataset supplies only `train: 0:206`; our held-out splits are custom research splits. Default: one of five outer test folds, with 10% of the remaining episodes reserved for validation. The notebook supports k ∈ {1,2,4,8}, H ∈ {1,8}, dropout, and reproducible seeds. It runs one configuration initially; full cross-validation, multi-seed ablations, simulator success/overlap, videos, and compounding-error studies remain later experiments.

In [ ]:
# Colab already provides a matched torch/torchvision pair; retain it.
%pip -q install "huggingface_hub>=0.34,<3" "pyarrow>=17,<26" "pandas>=2.2,<4" tqdm matplotlib

## 1. Requested imports
The original message's Markdown escape characters are removed so these are executable Python; every requested import and setting is retained.

In [ ]:
import time
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

# import standard PyTorch modules
import torch
import torch.nn as nn
import torch.optim as optim

# import torchvision module to handle image manipulation
import torchvision
import torchvision.transforms as transforms
print("Running on torch.__version__ = ", torch.__version__)

torch.backends.cudnn.deterministic = True

# utils
import os
from tqdm import tqdm
tqdm.pandas()
from collections import Counter

In [ ]:
import json
import copy
import hashlib
import platform
from pathlib import Path
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download
from torch.utils.data import Dataset, DataLoader
import pyarrow.parquet as pq

SEED = 42
FOLD = 0                      # 0..4; change only for a fresh outer-fold experiment
N_FOLDS = 5
VAL_FRACTION = 0.10           # fraction of non-test episodes
K = 4                        # history: 1, 2, 4, 8
H = 1                        # action chunk: 1 or 8
DROPOUT = 0.1                # set 0.0 for dropout ablation
EPOCHS = 10                   # starter budget, equal maximum for all architectures
PATIENCE = 3
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
OUT = Path("pusht_run")
OUT.mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert K in (1,2,4,8) and H in (1,8) and 0 <= FOLD < N_FOLDS
print("Device:", DEVICE)
# GPU kernels and library versions can still cause small cross-platform differences.

## 2. Download and inspect the verified parquet snapshot
Pin a commit so future dataset changes cannot silently change the experiment. The pandas equivalent of your original path is `pd.read_parquet("hf://datasets/lerobot/pusht@<revision>/data/chunk-000/file-000.parquet")`. Here we cache files first and discover **all** data shards, since a file is not necessarily an episode.

The public dataset needs no login. If access restrictions change, use `from huggingface_hub import notebook_login; notebook_login()`; never paste a token into a saved notebook.

In [ ]:
REPO_ID = "lerobot/pusht"
REVISION = "7628202a2180972f291ba1bc6723834921e72c19"
api = HfApi()
repo = api.dataset_info(REPO_ID, revision=REVISION)
assert repo.sha == REVISION
def download(name):
    return hf_hub_download(REPO_ID, name, repo_type="dataset", revision=REVISION)

with open(download("meta/info.json")) as f:
    info = json.load(f)
data_files = sorted(s.rfilename for s in repo.siblings
                    if s.rfilename.startswith("data/") and s.rfilename.endswith(".parquet"))
assert data_files, "No data parquet shards found"
paths = [download(name) for name in tqdm(data_files, desc="Parquet shards")]
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
print("Revision:", REVISION)
print("Format:", info["codebase_version"], "FPS:", info["fps"])
print("Official splits:", info["splits"])
print("Parquet shards:", data_files)
print("Rows / columns:", df.shape)
print(pq.read_schema(paths[0]))
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
display(pd.DataFrame(info["features"]).T)
print("Column names:", list(df.columns))
assert "observation.environment_state" not in df.columns
print("No block/keypoint state in this snapshot. Videos:", info.get("video_path"))

## 3. Integrity checks and episode inspection
Fail clearly on repeated, missing, nonfinite, or discontinuous samples instead of silently deleting frames and changing temporal alignment. Short episodes are retained: history is left-padded and unavailable future targets are masked.

In [ ]:
required = {"episode_index", "frame_index", "timestamp", "index", "observation.state", "action"}
assert required <= set(df.columns), f"Missing columns: {required - set(df.columns)}"
assert not df[list(required)].isna().any().any(), "Missing required values"
assert not df.duplicated(["episode_index", "frame_index"]).any(), "Repeated episode/frame"
assert df["index"].is_unique, "Repeated global index"
df = df.sort_values(["episode_index", "frame_index"]).reset_index(drop=True)
states = np.stack(df["observation.state"].to_numpy()).astype(np.float32)
actions = np.stack(df["action"].to_numpy()).astype(np.float32)
assert states.shape == (len(df), 2) and actions.shape == (len(df), 2)
assert np.isfinite(states).all() and np.isfinite(actions).all()
assert len(df) == info["total_frames"]
assert df.episode_index.nunique() == info["total_episodes"]
for ep, group in df.groupby("episode_index", sort=True):
    assert np.array_equal(group.frame_index.to_numpy(), np.arange(len(group))), f"Frame gap in {ep}"
    assert np.allclose(np.diff(group.timestamp), 1 / info["fps"], atol=1e-4), f"Timestamp gap in {ep}"

episode_table = df.groupby("episode_index").agg(frames=("frame_index", "size"),
                                               duration_s=("timestamp", "max"))
display(episode_table.describe())
print("Short episodes (< max(k,H)):", int((episode_table.frames < max(K,H)).sum()))
print("Episode lengths:", Counter(episode_table.frames).most_common(5))
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(episode_table.frames, bins=25)
ax[0].set(xlabel="Frames per episode", ylabel="Episodes")
first = df.episode_index.eq(df.episode_index.iloc[0]).to_numpy()
ax[1].plot(states[first,0], states[first,1], label="Pusher observation")
ax[1].plot(actions[first,0], actions[first,1], alpha=.6, label="Expert target")
ax[1].set(xlabel="x", ylabel="y", title="Example episode (raw coordinates)")
ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Leakage-safe five-fold episode partitions
Each outer fold is test once. Within the other four folds, reserve 10% of episodes for validation (rounded up). For 206 episodes, fold 0 has **147 training / 17 validation / 42 test episodes**. No frame-level random split. Reuse the same manifest across architectures, histories, horizons, and seeds; only vary the split seed for an explicitly different partition study.

In [ ]:
SPLIT_SEED = 42               # independent of the model/training seed
episode_ids = np.sort(df.episode_index.unique())
outer_folds = np.array_split(np.random.default_rng(SPLIT_SEED).permutation(episode_ids), N_FOLDS)
def split_for_fold(fold):
    test_ids = outer_folds[fold]
    pool = np.concatenate([ids for i, ids in enumerate(outer_folds) if i != fold])
    pool = np.random.default_rng(SPLIT_SEED + 1000 + fold).permutation(pool)
    nval = max(1, int(np.ceil(len(pool) * VAL_FRACTION)))
    return {"train": np.sort(pool[nval:]), "validation": np.sort(pool[:nval]),
            "test": np.sort(test_ids)}

all_splits = [split_for_fold(f) for f in range(N_FOLDS)]
for split in all_splits:
    sets = [set(ids.tolist()) for ids in split.values()]
    assert all(sets[i].isdisjoint(sets[j]) for i in range(3) for j in range(i+1,3))
    assert set.union(*sets) == set(episode_ids.tolist())
assert sorted(np.concatenate([s["test"] for s in all_splits]).tolist()) == episode_ids.tolist()
split_ids = all_splits[FOLD]
frame_split = np.full(len(df), "", dtype=object)
for name, ids in split_ids.items():
    frame_split[np.isin(df.episode_index.to_numpy(), ids)] = name
assert np.all(frame_split != "")
display(pd.DataFrame([{"split": n, "episodes": len(ids), "frames": int((frame_split==n).sum())}
                      for n, ids in split_ids.items()]))
manifest = {"repo": REPO_ID, "revision": REVISION, "split_seed": SPLIT_SEED,
            "validation_fraction": VAL_FRACTION,
            "folds": [{n: ids.tolist() for n, ids in s.items()} for s in all_splits]}
manifest_text = json.dumps(manifest, indent=2)
(OUT / "episode_splits.json").write_text(manifest_text)
split_hash = hashlib.sha256(manifest_text.encode()).hexdigest()
df[["index", "episode_index", "frame_index"]].assign(split=frame_split).to_csv(
    OUT / f"frame_assignments_fold{FOLD}.csv", index=False)
print("Manifest SHA256:", split_hash)

## 5. Train-only normalization and causal temporal samples
Use observations ending at t to predict expert actions **a[t:t+H]**, matching the proposal's equation (no off-by-one shift). The first observation repeats when t < k−1. Future targets beyond the episode are zero-padded **after normalization**, with zero loss weight. All three models see the same k observations. Set k=1 for the current-state MLP baseline, and compare models at equal k as well to separate history effects from architecture effects.

Fit statistics on raw training frames only. Do not use the repository's global `meta/stats.json`, which includes held-out episodes. Input features are explicitly whitelisted.

In [ ]:
train_mask = frame_split == "train"
obs_mean = states[train_mask].mean(0)
obs_std = np.maximum(states[train_mask].std(0), 1e-6)
act_mean = actions[train_mask].mean(0)
act_std = np.maximum(actions[train_mask].std(0), 1e-6)
norm = {"obs_mean": obs_mean, "obs_std": obs_std, "act_mean": act_mean, "act_std": act_std}
np.savez(OUT / f"normalization_fold{FOLD}.npz", **norm)
episodes = {}
for ep, group in df.groupby("episode_index", sort=True):
    idx = group.index.to_numpy()
    episodes[int(ep)] = ((states[idx] - obs_mean) / obs_std, (actions[idx] - act_mean) / act_std)

class EpisodeWindows(Dataset):
    def __init__(self, ids, k, horizon):
        self.k, self.horizon = k, horizon
        self.samples = [(int(ep), t) for ep in ids for t in range(len(episodes[int(ep)][0]))]
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        ep, t = self.samples[i]
        obs, act = episodes[ep]
        history_idx = np.maximum(np.arange(t-self.k+1, t+1), 0)
        x = obs[history_idx]
        count = min(self.horizon, len(act)-t)
        y = np.zeros((self.horizon, 2), dtype=np.float32)
        valid = np.zeros(self.horizon, dtype=np.float32)
        y[:count] = act[t:t+count]
        valid[:count] = 1
        return torch.from_numpy(x.copy()), torch.from_numpy(y), torch.from_numpy(valid), ep

datasets = {n: EpisodeWindows(ids, K, H) for n, ids in split_ids.items()}
def make_loader(name, shuffle=False):
    return DataLoader(datasets[name], batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, generator=torch.Generator().manual_seed(SEED))
for name, ds in datasets.items():
    assert {ep for ep,t in ds.samples} == set(split_ids[name].tolist())
    x,y,m,ep = ds[0]
    assert x.shape == (K,2) and y.shape == (H,2) and m.shape == (H,)
    assert torch.allclose(x, x[-1:].expand_as(x)) # episode-start history padding
    _,_,last_mask,_ = ds[len(ds)-1]
    assert last_mask.sum().item() == 1
print("Input / target / mask shapes:", x.shape, y.shape, m.shape)

## 6. Proposed policy networks (each under one million parameters)
MLP: flatten history, two hidden layers. LSTM: sequence → last hidden state. Transformer: learned positions, two encoder layers, four heads → last token. A causal attention mask is included; every input is already at or before t. LSTM state resets for every window, matching deployment with a sliding k-frame history. These use the same inputs, action head, loss, optimizer, and training budget; parameter counts are reported rather than claimed identical.

In [ ]:
class MLPPolicy(nn.Module):
    def __init__(self, k, horizon, dropout):
        super().__init__()
        self.horizon = horizon
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(k*2,128), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(128,128), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(128,horizon*2))
    def forward(self, x):
        return self.net(x).reshape(-1,self.horizon,2)

class LSTMPolicy(nn.Module):
    def __init__(self, k, horizon, dropout):
        super().__init__()
        self.horizon = horizon
        self.rnn = nn.LSTM(2,128,num_layers=2,batch_first=True,dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout),nn.Linear(128,horizon*2))
    def forward(self, x):
        sequence, _ = self.rnn(x)
        return self.head(sequence[:,-1]).reshape(-1,self.horizon,2)

class TransformerPolicy(nn.Module):
    def __init__(self, k, horizon, dropout):
        super().__init__()
        self.horizon = horizon
        self.embed = nn.Linear(2,64)
        self.position = nn.Parameter(torch.zeros(1,k,64))
        nn.init.normal_(self.position, std=.02)
        layer = nn.TransformerEncoderLayer(64,4,dim_feedforward=128,
                                           dropout=dropout,batch_first=True,activation="relu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.head = nn.Linear(64,horizon*2)
    def forward(self, x):
        z = self.embed(x) + self.position[:,:x.size(1)]
        mask = torch.triu(torch.ones(x.size(1),x.size(1),device=x.device,dtype=torch.bool),diagonal=1)
        z = self.encoder(z, mask=mask)
        return self.head(z[:,-1]).reshape(-1,self.horizon,2)

MODEL_TYPES = {"MLP": MLPPolicy, "LSTM": LSTMPolicy, "Transformer": TransformerPolicy}
for name, cls in MODEL_TYPES.items():
    model = cls(K,H,DROPOUT)
    nparams = sum(p.numel() for p in model.parameters())
    assert nparams < 1_000_000
    assert model(torch.zeros(3,K,2)).shape == (3,H,2)
    print(f"{name}: {nparams:,} parameters")

## 7. Train and select using validation only
Masked MSE is the mean squared error per **valid action coordinate**. Adam weight decay implements the proposal's L2 regularization. Early stopping restores the lowest-validation-loss weights; the test loader is not used here. This starter uses one seed and one fold. For the full study, rerun with fresh train-only statistics and models for each fold/seed/k/H/dropout setting, and report mean and standard deviation. Do not tune choices after seeing test metrics.

In [ ]:
def loss_parts(pred, target, mask):
    return (((pred-target)**2) * mask.unsqueeze(-1)).sum(), mask.sum()*target.shape[-1]

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    numerator = denominator = 0.0
    with torch.set_grad_enabled(training):
        for x,y,mask,_ in loader:
            x,y,mask = x.to(DEVICE),y.to(DEVICE),mask.to(DEVICE)
            if training: optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            s,n = loss_parts(pred,y,mask)
            loss = s/n
            assert torch.isfinite(loss), "Nonfinite loss"
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
            numerator += s.detach().item()
            denominator += n.detach().item()
    return numerator/denominator

trained, histories, checkpoint_paths = {}, {}, {}
config = {"seed": SEED, "fold": FOLD, "k": K, "horizon": H, "dropout": DROPOUT,
          "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
          "lr": LR, "weight_decay": WEIGHT_DECAY}
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__, "pandas": pd.__version__}
for name, cls in MODEL_TYPES.items():
    seed_everything(SEED)
    model = cls(K,H,DROPOUT).to(DEVICE)
    optimizer = optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    train_loader = make_loader("train",shuffle=True)
    val_loader = make_loader("validation")
    best_loss, best_state, stale = float("inf"), None, 0
    rows = []
    started = time.time()
    for epoch in range(1,EPOCHS+1):
        train_loss = run_epoch(model,train_loader,optimizer)
        val_loss = run_epoch(model,val_loader)
        rows.append({"epoch": epoch, "train_mse": train_loss, "validation_mse": val_loss})
        print(f"{name} {epoch:02d}: train={train_loss:.5f}, validation={val_loss:.5f}")
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE: break
    assert best_state is not None
    model.load_state_dict(best_state)
    trained[name] = model.eval()
    histories[name] = pd.DataFrame(rows)
    tag = f"{name}_fold{FOLD}_seed{SEED}_k{K}_H{H}_drop{DROPOUT}"
    checkpoint_paths[name] = OUT / f"{tag}.pt"
    torch.save({"model_type": name, "model_state": best_state, "config": config,
                "normalization": {key:value.tolist() for key,value in norm.items()},
                "revision": REVISION, "split_manifest_sha256": split_hash,
                "episode_ids": {key:value.tolist() for key,value in split_ids.items()},
                "best_epoch": best_epoch, "best_validation_mse": best_loss,
                "versions": versions, "seconds": time.time()-started}, checkpoint_paths[name])
    histories[name].to_csv(OUT / f"{tag}_history.csv",index=False)

fig, axes = plt.subplots(1,3,figsize=(14,3))
for ax,(name,history) in zip(axes,histories.items()):
    ax.plot(history.epoch,history.train_mse,label="Train")
    ax.plot(history.epoch,history.validation_mse,label="Validation")
    ax.set(title=name,xlabel="Epoch",ylabel="Normalized masked MSE")
    ax.legend()
plt.tight_layout(); plt.show()

## 8. Final offline test (run after fixing all choices)
Evaluate each restored checkpoint once. Report normalized and original-coordinate MSE, plus equal-weight episode MSE so longer episodes do not dominate every result. A train-mean action predictor provides a simple sanity baseline. Horizon curves below use expert observations and measure **offline action-chunk prediction**, not closed-loop compounding error or simulator success.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    if model is not None: model.eval()
    ns = rs = denom = 0.0
    ep_totals = {}
    horizon_sums = np.zeros(H)
    horizon_counts = np.zeros(H)
    scale = torch.tensor(act_std,device=DEVICE)
    for x,y,mask,eps in loader:
        x,y,mask = x.to(DEVICE),y.to(DEVICE),mask.to(DEVICE)
        pred = torch.zeros_like(y) if model is None else model(x)
        normalized_sq = (pred-y)**2
        raw_sq = normalized_sq * scale.square()
        ns += (normalized_sq*mask.unsqueeze(-1)).sum().item()
        rs += (raw_sq*mask.unsqueeze(-1)).sum().item()
        denom += (mask.sum()*2).item()
        per_sample_sum = (raw_sq*mask.unsqueeze(-1)).sum((1,2)).cpu().numpy()
        per_sample_n = (mask.sum(1)*2).cpu().numpy()
        for ep,s,n in zip(eps.tolist(),per_sample_sum,per_sample_n):
            old = ep_totals.get(ep,(0.,0.))
            ep_totals[ep] = (old[0]+float(s),old[1]+float(n))
        horizon_sums += (raw_sq*mask.unsqueeze(-1)).sum((0,2)).cpu().numpy()
        horizon_counts += (mask.sum(0)*2).cpu().numpy()
    return {"normalized_mse": ns/denom, "raw_coordinate_mse": rs/denom,
            "episode_mean_raw_mse": np.mean([s/n for s,n in ep_totals.values()])}, np.divide(
                horizon_sums,horizon_counts,out=np.full(H,np.nan),where=horizon_counts>0)

test_loader = make_loader("test")
results, horizon_curves = [], {}
for name,model in {"Train-mean baseline": None, **trained}.items():
    metrics, curve = evaluate(model,test_loader)
    results.append({"model": name, **metrics})
    horizon_curves[name] = curve
results = pd.DataFrame(results)
display(results)
results.to_csv(OUT / f"test_fold{FOLD}_seed{SEED}_k{K}_H{H}_drop{DROPOUT}.csv",index=False)
for name,curve in horizon_curves.items():
    plt.plot(np.arange(H),curve,marker="o",label=name)
plt.xlabel("Action offset from t (0 = current expert action)")
plt.ylabel("Raw-coordinate MSE per valid coordinate")
plt.title("Offline action-chunk error on held-out episodes")
plt.legend(); plt.show()

## 9. Reload a policy and predict one action chunk
This helper uses the same left-padding and normalization as training. Supply only observations up to the current time. In a future simulator integration, execute only the first predicted action and replan at the next step. The present state-only observation contract cannot observe the block, so a simulator rollout would not establish the full proposal's intended policy performance.

In [ ]:
# Load only your own trusted checkpoint.
selected_name = "MLP"  # choose explicitly; do not select architecture using test scores
checkpoint = torch.load(checkpoint_paths[selected_name],map_location="cpu",weights_only=True)
c = checkpoint["config"]
reloaded = MODEL_TYPES[checkpoint["model_type"]](c["k"],c["horizon"],c["dropout"])
reloaded.load_state_dict(checkpoint["model_state"])
reloaded.eval()

@torch.no_grad()
def predict_chunk(raw_history, model, checkpoint):
    k = checkpoint["config"]["k"]
    history = np.asarray(raw_history,dtype=np.float32)
    assert history.ndim == 2 and history.shape[1] == 2 and len(history)>0
    assert np.isfinite(history).all()
    history = history[-k:]
    if len(history)<k:
        history = np.concatenate([np.repeat(history[:1],k-len(history),axis=0),history])
    stats = {key: np.asarray(value,dtype=np.float32) for key,value in checkpoint["normalization"].items()}
    x = torch.from_numpy((history-stats["obs_mean"])/stats["obs_std"]).unsqueeze(0)
    pred = model(x.to(next(model.parameters()).device)).cpu().numpy()[0]
    return pred*stats["act_std"]+stats["act_mean"]

print("Predicted target chunk:", predict_chunk(states[:1],reloaded,checkpoint))
(OUT / "run_metadata.json").write_text(json.dumps({"config": config, "versions": versions,
    "revision": REVISION, "split_manifest_sha256": split_hash,
    "observation_contract": "pusher position only; no block state or images"},indent=2))
print("Saved outputs:")
for path in sorted(OUT.iterdir()): print(path)
# Optional Colab download (uncomment):
# import shutil
# from google.colab import files
# files.download(shutil.make_archive("pusht_run", "zip", OUT))

## Interpretation and next experiments
* The runnable three-policy comparison is a starting point. Pusher position alone cannot uniquely determine block pose; do not interpret its errors as a comparison using full environment state.
* To complete the proposal's observation design, verify a separate keypoint dataset or add an image encoder with correctly synchronized videos and identical episode assignments. Do not merge in guessed 16D features.
* For full five-fold evaluation, rerun sections 4–9 for FOLD=0…4, fitting normalization afresh each time. Keep split seed fixed; use training seeds such as 0, 1, 2. Start with k=1/H=1, then k=2,4,8 and H=8; dropout 0 or 0.1. Aggregate fold/seed results rather than choosing a configuration by its outer-test score. This notebook does not automatically execute the full grid.
* Closed-loop overlap/success over 50 random starts, videos, and compounding-error measurements need a verified `gym-pusht` observation/action adapter and simulator runs. They are not supplied by offline MSE and have not been run here.
* For H=8, overlapping training chunks mean an action can appear in several targets **within the same split**. No target or observation window crosses an episode boundary.

### Verified sources
* [Pinned machine-readable dataset metadata](https://huggingface.co/datasets/lerobot/pusht/blob/7628202a2180972f291ba1bc6723834921e72c19/meta/info.json) — authoritative for this snapshot; the README still illustrates an older v2 layout.
* [Pinned parquet data](https://huggingface.co/datasets/lerobot/pusht/tree/7628202a2180972f291ba1bc6723834921e72c19/data/chunk-000) — actual Arrow schema and all 25,650 rows were checked during notebook preparation.
* [Dataset card](https://huggingface.co/datasets/lerobot/pusht) and [LeRobot source](https://github.com/huggingface/lerobot).
* Local proposal: *Comparing MLP, LSTM, and Transformer Policies for Robot Imitation Learning on PushT*, UBC EECE 568, pp. 1–2. Architectures, k/H ablations, padding/masking, five-fold design, Adam, and early stopping follow that proposal; the missing block-state limitation is a verified correction.

**Preparation validation:** notebook schema and Python syntax passed. The complete workflow ran locally on the pinned 25,650-row dataset with k=4, H=8 and one epoch for each of MLP, LSTM, and Transformer, including validation selection, test metrics, saved checkpoints, and checkpoint reload. Boundary alignment/padding/masking checks passed, as did forward/backward checks for every proposed k/H pair. The delivered notebook retains its starter defaults (10 epochs, k=4, H=1) and clean outputs for your own Colab run. The full study and simulator evaluation have not been run.